<a href="https://colab.research.google.com/github/vinodr-intern/prodigyinfotechinternship/blob/main/task_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!rm -rf *

In [7]:
!wget https://github.com/ardamavi/Sign-Language-Digits-Dataset/archive/refs/heads/master.zip
!unzip master.zip

--2026-03-26 05:48:03--  https://github.com/ardamavi/Sign-Language-Digits-Dataset/archive/refs/heads/master.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/ardamavi/Sign-Language-Digits-Dataset/zip/refs/heads/master [following]
--2026-03-26 05:48:03--  https://codeload.github.com/ardamavi/Sign-Language-Digits-Dataset/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 140.82.112.10
Connecting to codeload.github.com (codeload.github.com)|140.82.112.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘master.zip’

master.zip              [    <=>             ]  16.43M  23.2MB/s    in 0.7s    

2026-03-26 05:48:04 (23.2 MB/s) - ‘master.zip’ saved [17231669]

Archive:  master.zip
4b09433ef9d5c60602e8805bb54b8aa2428527f8
   creating: Sign-L

In [8]:
DATASET_PATH = "Sign-Language-Digits-Dataset-master"

In [10]:
import os
import cv2
import numpy as np
from skimage.feature import hog

def load_data(base_path, limit_per_class=150):
    X, y = [], []

    for person in os.listdir(base_path):
        person_path = os.path.join(base_path, person)

        if not os.path.isdir(person_path):
            continue

        for gesture in os.listdir(person_path):
            gesture_path = os.path.join(person_path, gesture)

            count = 0

            for file in os.listdir(gesture_path):
                if count >= limit_per_class:
                    break

                try:
                    img_path = os.path.join(gesture_path, file)
                    img = cv2.imread(img_path)

                    if img is None:
                        continue

                    img = cv2.resize(img, (64, 64))
                    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                    features = hog(
                        gray,
                        orientations=9,
                        pixels_per_cell=(8, 8),
                        cells_per_block=(2, 2)
                    )

                    X.append(features)
                    y.append(gesture)
                    count += 1

                except:
                    continue

    return np.array(X), np.array(y)

In [12]:
import os
print(os.listdir())

['.config', 'master.zip', 'Sign-Language-Digits-Dataset-master']


In [13]:
DATASET_PATH = "Sign-Language-Digits-Dataset-master/Dataset"

In [14]:
import os
import cv2
import numpy as np

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [15]:
def load_data(base_path, limit_per_class=200):
    X, y = [], []

    for label_folder in os.listdir(base_path):
        folder_path = os.path.join(base_path, label_folder)

        if not os.path.isdir(folder_path):
            continue

        count = 0

        for file in os.listdir(folder_path):
            if count >= limit_per_class:
                break

            img_path = os.path.join(folder_path, file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (64, 64))
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            features = hog(
                gray,
                orientations=9,
                pixels_per_cell=(8, 8),
                cells_per_block=(2, 2)
            )

            X.append(features)
            y.append(label_folder)
            count += 1

    return np.array(X), np.array(y)


X, y = load_data(DATASET_PATH)
print("Shape:", X.shape)

Shape: (2000, 1764)


In [16]:
le = LabelEncoder()
y = le.fit_transform(y)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = SVC(kernel='rbf', C=10)
model.fit(X_train, y_train)

SVC(C=10)

In [18]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8825
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        48
           1       0.94      0.98      0.96        46
           2       0.79      0.79      0.79        34
           3       0.95      0.93      0.94        44
           4       0.74      0.85      0.79        33
           5       1.00      0.97      0.99        36
           6       0.71      0.85      0.77        34
           7       0.86      0.76      0.81        42
           8       0.88      0.77      0.82        47
           9       0.92      0.92      0.92        36

    accuracy                           0.88       400
   macro avg       0.88      0.88      0.88       400
weighted avg       0.89      0.88      0.88       400



In [19]:
import joblib

joblib.dump(model, "gesture_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [29]:
gesture_names = {
    '0': 'Zero ✊',
    '1': 'One ☝️',
    '2': 'Two ✌️',
    '3': 'Three 🤟',
    '4': 'Four 🖖',
    '5': 'Five ✋',
    '6': 'Six 🤙',
    '7': 'Seven 👆',
    '8': 'Eight ✊✌️',
    '9': 'Nine 🤟✋'
}

def predict_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        return "Invalid image"

    img = cv2.resize(img, (64, 64))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )

    features = features.reshape(1, -1)
    features = scaler.transform(features)

    pred = model.predict(features)

    label = le.inverse_transform(pred)[0]

    return gesture_names[label]   # ✅ inside function

In [30]:
print(predict_image("Sign-Language-Digits-Dataset-master/Dataset/0/IMG_1118.JPG"))

Zero ✊


In [24]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[47  0  0  0  0  0  0  0  1  0]
 [ 0 45  1  0  0  0  0  0  0  0]
 [ 0  1 27  0  0  0  6  0  0  0]
 [ 0  0  2 41  1  0  0  0  0  0]
 [ 0  0  0  0 28  0  3  1  1  0]
 [ 0  0  0  1  0 35  0  0  0  0]
 [ 0  1  2  0  2  0 29  0  0  0]
 [ 0  1  1  0  4  0  2 32  2  0]
 [ 0  0  0  0  3  0  1  4 36  3]
 [ 0  0  1  1  0  0  0  0  1 33]]
